In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

# 6 seeds that gave LB 0.37468 + 3 new ones = 9 total
SEEDS    = [42, 7, 123, 13, 99, 2024, 17, 777, 31]
N_SPLITS = 10
print(f'Seeds: {SEEDS}  ({len(SEEDS)} total)')

Seeds: [42, 7, 123, 13, 99, 2024, 17, 777, 31]  (9 total)


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')

Train: (13249, 41), Test: (8834, 41)


In [4]:
# ── Cell 4: Preprocessing ────────────────────────────────────────────────────
def preprocess(df):
    df = df.copy()
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)
    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    df = df.fillna(-1)
    return df


X_train_raw = preprocess(TRAIN_DATA)
X_test_raw  = preprocess(TEST_DATA)
y_train     = TRAIN_LABEL['disorder'].values

# Same adversarial feature drops as best run
DROP_ADV = ['blood_cell_count', 'white_blood_cell_count', 'mother_age']
X_train = X_train_raw.drop(columns=DROP_ADV)
X_test  = X_test_raw.drop(columns=DROP_ADV)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 56), X_test: (8834, 56)


In [5]:
# ── Cell 5: Class weights ────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

# Best Optuna params — locked
BEST_PARAMS = {
    'depth'              : 8,
    'l2_leaf_reg'        : 1.2554074561515052,
    'random_strength'    : 0.15300699009014024,
    'rsm'                : 0.6693037260566354,
    'bagging_temperature': 0.7561917100518147,
    'min_data_in_leaf'   : 23,
}
print('Class weights and params ready.')
print(f'BEST_PARAMS: {BEST_PARAMS}')

Class weights and params ready.
BEST_PARAMS: {'depth': 8, 'l2_leaf_reg': 1.2554074561515052, 'random_strength': 0.15300699009014024, 'rsm': 0.6693037260566354, 'bagging_temperature': 0.7561917100518147, 'min_data_in_leaf': 23}


In [6]:
# ── Cell 6: Train — 9 seeds x 10 folds ───────────────────────────────────────
print(f'Training {len(SEEDS)} seeds x {N_SPLITS} folds = {len(SEEDS)*N_SPLITS} models total')

all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test), 10))
seed_oof_scores = []

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx],       y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            class_weights         = class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
            **BEST_PARAMS
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')
        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    seed_oof_scores.append(oof_score)
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'Per-seed OOF: {[round(s,4) for s in seed_oof_scores]}')
print(f'Seed std:     {np.std(seed_oof_scores):.4f}  (lower = more stable)')
print(f'FINAL OOF BA ({len(SEEDS)} seeds): {final_oof:.4f}')
print(f'6-seed OOF:                        0.3949  (LB 0.37468)')
print(f'Change:                            {final_oof - 0.3949:+.4f}')
print(f'Predicted LB ≈ OOF − 0.020:        {final_oof - 0.020:.4f}')
print(f"{'='*60}")

Training 9 seeds x 10 folds = 90 models total

======================================== SEED=42 ========================================
  Fold  1: BA=0.3718  best_iter=44
  Fold  2: BA=0.4215  best_iter=23
  Fold  3: BA=0.4189  best_iter=71
  Fold  4: BA=0.4042  best_iter=20
  Fold  5: BA=0.4142  best_iter=87
  Fold  6: BA=0.4338  best_iter=38
  Fold  7: BA=0.3609  best_iter=2
  Fold  8: BA=0.3756  best_iter=19
  Fold  9: BA=0.3952  best_iter=17
  Fold 10: BA=0.4357  best_iter=3
  OOF BA (seed=42): 0.4032 | mean=0.4032 ± 0.0251

======================================== SEED=7 ========================================
  Fold  1: BA=0.4138  best_iter=53
  Fold  2: BA=0.4124  best_iter=11
  Fold  3: BA=0.4125  best_iter=95
  Fold  4: BA=0.4191  best_iter=76
  Fold  5: BA=0.3673  best_iter=28
  Fold  6: BA=0.4197  best_iter=62
  Fold  7: BA=0.3766  best_iter=6
  Fold  8: BA=0.3661  best_iter=4
  Fold  9: BA=0.4156  best_iter=24
  Fold 10: BA=0.3863  best_iter=66
  OOF BA (seed=7): 0.3993 |

In [7]:
# ── Cell 7: Per-class recall ──────────────────────────────────────────────────
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
best_recall = {0:0.411, 1:0.362, 2:0.296, 3:0.319, 4:0.759,
               5:0.309, 6:0.503, 7:0.253, 8:0.560, 9:0.175}

oof_labels = np.argmax(all_oof_proba, axis=1)
report     = classification_report(y_train, oof_labels, output_dict=True)

print(f'OOF BA: {final_oof:.4f}  (6-seed best: 0.3949 → LB 0.37468)\n')
print(f'{"Class":<5} {"Name":<16} {"6 seeds":>10} {"9 seeds":>8} {"Δ":>7}')
print('-' * 52)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = best_recall[cls]
    delta = r - r_old
    flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>10.3f} {r:>8.3f} {delta:>+7.3f}{flag}')

OOF BA: 0.3904  (6-seed best: 0.3949 → LB 0.37468)

Class Name                6 seeds  9 seeds       Δ
----------------------------------------------------
0     레베르시                  0.411    0.411  +0.000
1     낭포성섬유증                0.362    0.362  +0.000
2     당뇨                    0.296    0.301  +0.005
3     리증후군                  0.319    0.325  +0.006
4     암                     0.759    0.707  -0.052
5     테이-삭스                 0.309    0.309  +0.000
6     혈색소침착증                0.503    0.507  +0.004
7     사립체근병종                0.253    0.251  -0.002 ← LOW
8     알츠하이머                 0.560    0.560  +0.000
9     확인안됨                  0.175    0.171  -0.004 ← LOW


In [8]:
# ── Cell 8: Save submission ───────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_9seeds.csv')

print('Saved: submission_9seeds.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nFinal OOF BA:    {final_oof:.4f}')
print(f'Predicted LB:    {final_oof - 0.020:.4f}  (using −0.020 gap)')
print(f'Current best LB: 0.37468')
print(f'Submit if predicted LB > 0.37468 AND seed std < 0.004')

Saved: submission_9seeds.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     485
1    1265
2     707
3    1454
4     233
5    1169
6    1059
7    1201
8     184
9    1077
Name: count, dtype: int64

Final OOF BA:    0.3904
Predicted LB:    0.3704  (using −0.020 gap)
Current best LB: 0.37468
Submit if predicted LB > 0.37468 AND seed std < 0.004
